Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Steered Molecular Dynamics (SMD) Simulations

This notebook prepares the initial structures for each window of **Umbrella Sampling** by performing **Steered Molecular Dynamics (SMD)** on the helical structure of deca-alanine.

SMD is a method that induces structural changes by applying external forces to molecules or atoms.
The calculation is carried out following these steps:

1. **Definition of Collective Variable (CV):**
   The distance between atoms at both ends of the molecule is defined as the collective variable (reaction coordinate).
2. **Moving Restraint:**
   A harmonic oscillator (spring) restraint is applied to the defined collective variable. By gradually changing the restraint position over time, the molecule or atoms are pulled/pushed to force a structural transition.

**Note:** This notebook only works in an environment where PLUMED is installed.


## Step 1. PLUMED Environment Configuration

PLUMED, used for Steered MD, is an external library. Paths and environment variables must be configured so the Python kernel can call it correctly.

**Note:** Please modify the `PLUMED_ROOT` path according to your specific environment.

In [ ]:
# PLUMED environment
import os
import sys

# Path setting for PLUMED
PLUMED_ROOT   = os.path.expanduser("~/local/plumed-2.9.0")  # 必要に応じてplumedをインストールしたdirに修正する
plumed_bin    = os.path.join(PLUMED_ROOT, "bin")
plumed_lib    = os.path.join(PLUMED_ROOT, "lib")
plumed_kernel = os.path.join(plumed_lib, "libplumedKernel.so")

# Environment variable settings
os.environ["PATH"]            = f"{plumed_bin}:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{plumed_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["PLUMED_KERNEL"]   = str(plumed_kernel)

# Add PLUMED to Python library path
if str(PLUMED_ROOT) not in sys.path:
    sys.path.append(str(PLUMED_ROOT))

## Step 2: Import Libraries and Configure PFP

In [ ]:
import numpy as np

# ASE
from ase import units
from ase.io import write
from ase.io.proteindatabank import read_proteindatabank
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.calculators.plumed import Plumed

# PFP
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

calc_mode = "R2SCAN_PLUS_D3"
method_type = "PFVM_D3_PFVM"
model_version = "v8.0.0"
estimator = Estimator(calc_mode=calc_mode, method_type=method_type, model_version=model_version)
calculator = ASECalculator(estimator)

## Step 3. Loading the Initial Structure

We load the equilibrated structure `deca_alanine_helix_eq.pdb` from the previous notebook [(Molecular Dynamics Simulation of NVT Ensemble)](./02_equilibrium_nvt_md_en.ipynb). If the equilibration process was skipped or the file is missing, the reference structure in `assets` will be used.

We also create the output directory (`./output/03_steered_md`) for this notebook.

In [ ]:
# Input PDB file
if os.path.exists(f"./output/02_equilibrium_nvt_md/deca_alanine_helix_eq.pdb"): # 02平衡化MDを実施している場合
    inp_pdb = "./output/02_equilibrium_nvt_md/deca_alanine_helix_eq.pdb"
else:
    inp_pdb = f"./assets/03_steered_md/deca_alanine_helix_eq.pdb"

# Output directory
out_root = './output/03_steered_md'
os.makedirs(out_root, exist_ok=True)

In [ ]:
# PDBを読み込む
atoms = read_proteindatabank(inp_pdb)
atoms

## Step 4. SMD Schedule Configuration (Target Distances)

Based on the end-to-end atomic distance of the loaded initial structure (equilibrium state), we vary the target distance in two directions: compression and extension.

* **Compression** (`cv_compression`): Shortening the end-to-end distance to reach a collapsed helix state.
* **Extension** (`cv_stretch`): Extending the end-to-end distance to reach a fully stretched chain state.

Here, the distance is changed in increments of 0.05 Å (`d_step`). It is crucial to change the distance in small steps to avoid system failure or unexpected structural transition paths.

In [ ]:
# Steered MD control parameters
d_step  = 0.05   # Distance step size [Å]

# Atom indices (controlling the distance between CA atoms at both ends)
atom_idx_1 = 8   # N-terminal CA atom
atom_idx_2 = 98  # C-terminal CA atom

# Calculate the starting distance
d_start = round(atoms.get_distance(atom_idx_1, atom_idx_2) / d_step) * d_step

# Define the range of exploration
d_min   = 13.0
d_max   = 38.0

# Time steps for each window
steps_window = 1_000

# Compression: from d_start to d_min
num_comp = int(round((d_start - d_min) / d_step)) + 1
cv_compression = np.linspace(d_start, d_min, num=num_comp)

# Stretch: from d_start + d_step to d_max
num_stretch = int(round((d_max - (d_start + d_step)) / d_step)) + 1
cv_stretch = np.linspace(d_start + d_step, d_max, num=num_stretch)

print(f"Start Distance: {d_start:.2f} A")
print(f"Range: {d_min} A <-> {d_max} A")

## Step 5. Execution of Steered MD
### 5-1. Steered MD: Compression Direction

Simulations are performed continuously in the compression direction according to the defined schedule (`cv_compression`).

* **PLUMED Configuration:**
  * `DISTANCE` command defines the distance between two atoms .
  * `RESTRAINT` command applies a harmonic potential with a spring constant (`KAPPA`) at the center value (`AT`).
  * `UNITS` command set unit to eV and Å.

* **Simulation Flow:**
  1. Perform 1,000 steps of Langevin dynamics at each target distance (`AT)`.
  2. Save the final structure as `md-dyn-restart.pdb`.
  3. In the next loop, the final structure of the previous window is used as the initial structure (the same `atoms` object is updated continuously).

In [ ]:
for cv in cv_compression:

    # ------------------------------------------------------------
    # Prepare Directory
    # ------------------------------------------------------------
    cv_str = f"{cv:.2f}"
    out_dir = f"{out_root}/cv_{cv_str}"
    os.makedirs(out_dir, exist_ok=True)

    # ------------------------------------------------------------
    # PLUMED Settingsof Collective Variables
    # ------------------------------------------------------------
    # PLUMED uses 1-based index, so add +1 to Python index
    plumed_idx_1 = atom_idx_1 + 1
    plumed_idx_2 = atom_idx_2 + 1

    plumed_setting = [
        f"UNITS LENGTH=A ENERGY=eV",

        # 1. Define distance
        f"dist: DISTANCE ATOMS={plumed_idx_1},{plumed_idx_2}",

        # 2. Apply umbrella potential (https://www.plumed.org/doc-v2.9/user-doc/html/lugano-2.html)
        f"restraint: RESTRAINT ARG=dist KAPPA=0.2 AT={cv_str}",

        # 3. Output
        f"PRINT STRIDE=100 ARG=dist,restraint.bias,restraint.force2 FILE={out_dir}/COLVAR_{cv_str}",
        "FLUSH STRIDE=1000"
    ]

    print(plumed_setting)

    # ------------------------------------------------------------
    # Molecular Dynamics
    # ------------------------------------------------------------
    timestep = 1.0 * units.fs
    temperature = 300.0

    # PLUMED calculator
    atoms.calc = Plumed(calc=calculator, input=plumed_setting, timestep=timestep, atoms=atoms, kT=1)

    # Set the momenta corresponding to the given "temperature"
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature,force_temp=True)
    Stationary(atoms)  # Set zero total momentum to avoid drifting

    # Dynamics
    dyn = Langevin(atoms,
                   timestep,
                   temperature_K=temperature,
                   friction=0.002/units.fs,
                   trajectory=f'{out_dir}/md-dyn.traj',
                   logfile=f'{out_dir}/md-dyn.log',
                   loginterval=100)

    dyn.run(steps_window)

    write(f'{out_dir}/md-dyn-restart.pdb', atoms)

### 5-2. Steered MD: Extension Direction

Simulations are performed to pull the molecule in the extension direction according to the schedule (`cv_stretch`).


In [ ]:
# Reload initial structure
atoms = read_proteindatabank(inp_pdb)

for cv in cv_stretch:

    # ------------------------------------------------------------
    # Prepare Directory
    # ------------------------------------------------------------
    cv_str = f"{cv:.2f}"
    out_dir = f"{out_root}/cv_{cv_str}"
    os.makedirs(out_dir, exist_ok=True)

    # ------------------------------------------------------------
    # PLUMED Settings　of Collective Variables
    # ------------------------------------------------------------

    # PLUMED uses 1-based index, so add +1 to Python index
    plumed_idx_1 = atom_idx_1 + 1
    plumed_idx_2 = atom_idx_2 + 1

    plumed_setting = [
        f"UNITS LENGTH=A ENERGY=eV",

        # 1. Define distance
        f"dist: DISTANCE ATOMS={plumed_idx_1},{plumed_idx_2}",

        # 2. Apply umbrella potential (https://www.plumed.org/doc-v2.9/user-doc/html/lugano-2.html)
        f"restraint: RESTRAINT ARG=dist KAPPA=0.2 AT={cv_str}",

        # 3. Output
        f"PRINT STRIDE=100 ARG=dist,restraint.bias,restraint.force2 FILE={out_dir}/COLVAR_{cv_str}",
        "FLUSH STRIDE=1000"
    ]

    print(plumed_setting)

    # ------------------------------------------------------------
    # Molecular Dynamics
    # ------------------------------------------------------------
    timestep = 1.0 * units.fs
    temperature = 300.0

    # PLUMED calculator
    atoms.calc = Plumed(calc=calculator, input=plumed_setting, timestep=timestep, atoms=atoms, kT=1)

    # Set the momenta corresponding to the given "temperature"
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature,force_temp=True)
    Stationary(atoms)  # Set zero total momentum to avoid drifting

    # Dynamics
    dyn = Langevin(atoms,
                   timestep,
                   temperature_K=temperature,
                   friction=0.002/units.fs,
                   trajectory=f'{out_dir}/md-dyn.traj',
                   logfile=f'{out_dir}/md-dyn.log',
                   loginterval=100)

    dyn.run(steps_window)

    write(f'{out_dir}/md-dyn-restart.pdb', atoms)

### Next Step

Through Steered MD, we have obtained a dataset of structures varying continuously from compressed to extended states.
In the next notebook [(04_select_umbrella_sampling_initial_structures)](./04_select_umbrella_sampling_initial_structures_en.ipynb), we will extract the structures most suitable for Umbrella Sampling calculations from the generated trajectories.